# Business Intake Demo（D01 / D11）

> Issue #10

演示 `wanglaoshi.business`：接收 CSV → 只读归档 + sha256 → 缺列警告 → 阻断时跳过下游。


In [ ]:
from pathlib import Path
import pandas as pd
from wanglaoshi import BusinessPipeline, intake_file

demo = Path('outputs/business_demo')
demo.mkdir(parents=True, exist_ok=True)
src = demo / 'orders.csv'
pd.DataFrame({'order_id': [1, 2, 3], 'amount': [10.5, 20.0, 7.2], 'channel': ['web', 'app', 'web']}).to_csv(src, index=False)

result = intake_file(src, demo / 'archive', required_columns=['order_id', 'amount'], period='2024-Q1')
print(result.status)
print(result.data)


In [ ]:
pipe = BusinessPipeline(demo / 'archive_run', required_columns=['order_id', 'amount', 'channel'])
record = pipe.run(src)
print(record.summary())


In [ ]:
# 缺列 → warning；缺文件 → blocked 且下游 skipped
warn = intake_file(src, demo / 'archive2', required_columns=['order_id', 'amount', 'missing_col'])
print('warn', warn.status, warn.warnings)

blocked = BusinessPipeline(demo / 'archive3').run(demo / 'no_such.csv')
print('blocked', blocked.summary())
